In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from pathlib import Path

# load results
segment_risk = pd.read_csv("../data/processed/segment_risk.csv")
roads = gpd.read_file("../data/external/nyc_roads.gpkg")

# merge risk back onto road geometries
roads['physicalid'] = roads['physicalid'].astype(str)
segment_risk['physicalid'] = segment_risk['physicalid'].astype(str)
roads_risk = roads.merge(segment_risk, on='physicalid', how='inner')
roads_risk = roads_risk.to_crs("EPSG:4326")

print(f"Roads with risk scores: {len(roads_risk):,}")

# color map
color_map = {'Low': 'green', 'Moderate': 'orange', 'High': 'red'}

# build folium map centered on NYC
m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB positron')

# plot a sample of roads (plotting all 60k will freeze browser)
sample = roads_risk.sample(n=5000, random_state=42)

for _, row in sample.iterrows():
    if row.geometry is None:
        continue
    color = color_map.get(str(row['risk_class']), 'gray')
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda x, c=color: {
            'color': c, 'weight': 2, 'opacity': 0.7
        }
    ).add_to(m)

# legend
legend = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 10px; border-radius: 8px; font-size: 14px;">
     <b>Collision Risk</b><br>
     <span style="color:green">●</span> Low<br>
     <span style="color:orange">●</span> Moderate<br>
     <span style="color:red">●</span> High
</div>
"""
m.get_root().html.add_child(folium.Element(legend))

Path("../outputs/maps").mkdir(parents=True, exist_ok=True)
m.save("../outputs/maps/nyc_risk_map.html")
print("Map saved to outputs/maps/nyc_risk_map.html")